In [ ]:
# === Cell 1: Install Dependencies ===
# تثبيت المكتبات المطلوبة / Install required libraries
!apt-get -y install poppler-utils tesseract-ocr tesseract-ocr-ara -qq
!pip install -q pdf2image easyocr pyspellchecker langdetect transformers peft \
    huggingface_hub datasets Pillow opencv-python-headless pandas numpy \
    matplotlib scikit-learn pytesseract arabic-reshaper python-bidi \
    ar-corrector "gradio>=4.0" openpyxl jiwer torch torchvision \
    tensorboard albumentations tqdm uvicorn fastapi pyngrok

In [ ]:
# === Cell 2: Clone Project and Setup ===
# استنساخ المشروع وإعداد المسارات / Clone repo and setup paths
import os
REPO_URL = "https://github.com/DrAbdulmalek/HandwrittenOCR"
PROJECT_DIR = "/content/HandwrittenOCR"
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

In [ ]:
# === Cell 3: Mount Google Drive ===
# ربط Google Drive للحصول على ملفات PDF وحفظ النتائج
from google.colab import drive, userdata
drive.mount('/content/drive')

# جلب HF_TOKEN من Colab Secrets
# Get HF_TOKEN from Colab Secrets
HF_TOKEN = ''
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded')
except Exception:
    print('No HF_TOKEN')

# إنشاء مجلد المشروع على Drive / Create project folder on Drive
import os, shutil
from pathlib import Path
DRIVE_ROOT = '/content/drive/MyDrive/Handwritten_OCR_Ultimate'
if not os.path.exists(DRIVE_ROOT):
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('Drive folder created')

In [ ]:
# === Cell 4: Configure and Setup ===
# إعداد الإعدادات وتهيئة المشروع / Initialize config and project setup
import sys
sys.path.insert(0, '/content/HandwrittenOCR')

from config import Config
cfg = Config.from_colab_drive(
    pdf_name='python notes.pdf',
    hf_token=HF_TOKEN or '',
)
cfg.setup()
cfg.setup_easyocr_symlink()
print(f'Config ready: {cfg.root}')
print(f'Colab mode: {cfg.is_colab}')
print(f'Sync enabled: {cfg.sync_enabled}')

In [ ]:
# === Cell 5: Start API Server (FastAPI on Colab) ===
# بدء خادم API مع ngrok tunnel للوصول الخارجي
# Start FastAPI server with ngrok tunnel for external access
import threading
import subprocess
import time

# Start FastAPI server
PORT = 8000
server_cmd = f"cd /content/HandwrittenOCR && python -m uvicorn backend.app:app --host 0.0.0.0 --port {PORT} --no-reload"
proc = subprocess.Popen(server_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)
print(f'API server started on port {PORT}')

# Setup ngrok tunnel
from pyngrok import ngrok
try:
    public_url = ngrok.connect(PORT)
    print(f'Public URL: {public_url}')
    BASE_URL = str(public_url).replace('"', '')
except Exception as e:
    print(f'ngrok error: {e}')
    BASE_URL = f'http://localhost:{PORT}'
    print(f'Use port forwarding instead')

In [ ]:
# === Cell 5b: Data Migration (اختياري - شغّله مرة واحدة) ===# ترحيل البيانات من النسخ القديمة للمشروع# يشمل: قواعد بيانات موثقة + تصحيحات + قاموس التصحيحfrom src.migration import DataMigratormigrator = DataMigrator(cfg)# الخطوة 1: فحص المشاريع القديمةreport = migrator.scan_and_report()print(f"المشاريع المكتشفة: {report["projects_found"]}")for p in report['projects']:    print(f'  {p["name"]}: {len(p.get("db_files", []))} DB + {len(p.get("feedback_files", []))} feedback')# الخطوة 2: تشغيل الترحيل (احذف # لتفعيله)# result = migrator.migrate(#     base_path='/content/drive/MyDrive',#     verified_only=True,# )# print(f"تم ترحيل: {result["db_records_migrated"]} كلمة + {result["feedback_merged"]} تصحيح + {result["correction_dict_entries"]} قاموس")# أو عبر API بعد بدء الخادم:# r = requests.get(f'{BASE_URL}/api/migration/scan')# r = requests.post(f'{BASE_URL}/api/migration/run', json={'verified_only': True})print('Migration ready - uncomment the migrate() call above to run')

In [ ]:
# === Cell 6: Usage Examples ===
# أمثلة على استخدام API / API usage examples
import requests

# Health check / فحص حالة الخادم
r = requests.get(f"{BASE_URL}/api/health")
print("Health:", r.json())

# Process PDF / معالجة ملف PDF
r = requests.post(f"{BASE_URL}/api/process-pdf", json={
    "pdf_path": "/content/drive/MyDrive/python notes.pdf",
    "pages_start": 1,
    "pages_end": 5,
})
print("Process:", r.json())

In [ ]:
# === Cell 7: Status Queries ===
# استعلامات الحالة / Status queries

# Check processing stats / فحص إحصائيات المعالجة
r = requests.get(f"{BASE_URL}/api/stats")
print("Stats:", r.json())

# Status counts / عدد الحالات
r = requests.get(f"{BASE_URL}/api/status-counts")
print("Counts:", r.json())

# Check checkpoint / فحص نقطة الاستئناف
r = requests.get(f"{BASE_URL}/api/checkpoint")
print("Checkpoint:", r.json())

# Sync status (v5.0) / حالة المزامنة
r = requests.get(f"{BASE_URL}/api/sync/status")
print("Sync:", r.json())

In [ ]:
# === Cell 8: Review and Export ===
# المراجعة والتصدير / Review and export

# Get unverified words / استرجاع الكلمات غير الموثقة
r = requests.get(f"{BASE_URL}/api/words", params={"status": "unverified", "limit": 10})
data = r.json()
print(f"Unverified words: {data['total']}")
for w in data['words']:
    print(f"  [{w['image_id']}] {w['predicted_text']} ({w['confidence']:.2%})")

# Export dataset / تصدير بيانات التدريب
r = requests.post(f"{BASE_URL}/api/export-dataset", json={"val_ratio": 0.1})
print("Export:", r.json())

In [ ]:
# === Cell 9: Metrics and Analysis ===
# المقاييس والتحليل / Metrics and analysis

# Compute WER/CER / حساب WER/CER
r = requests.get(f"{BASE_URL}/api/metrics")
print("Metrics:", r.json())

# Metrics history / سجل المقاييس
r = requests.get(f"{BASE_URL}/api/metrics/history")
print("History:", r.json())

# Bilingual vocab / القاموس ثنائي اللغة
r = requests.get(f"{BASE_URL}/api/bilingual-vocab")
print("Bilingual vocab:", r.json())

# Network info (for mobile access) / معلومات الشبكة
r = requests.get(f"{BASE_URL}/api/network")
print("Network:", r.json())

In [ ]:
# === Cell 10: Local Mode Guide (Manjaro/Windows/macOS) ===
# دليل التشغيل المحلي - انسخ هذه الأوامر إلى الطرفية على جهازك
#
# 1. تثبيت اعتماديات النظام (Manjaro):
#    sudo pacman -S tesseract tesseract-data-ara poppler python python-pip base-devel
#
# 2. إنشاء بيئة افتراضية وتثبيت المكتبات:
#    cd ~/Handwritten_OCR_Ultimate
#    python -m venv ocr_env
#    source ocr_env/bin/activate
#    pip install -r requirements.txt
#
# 3. تشغيل المشروع محلياً:
#    python run.py --local
#    # أو مع تحديد ملف PDF:
#    python run.py --local --pdf ~/documents/notes.pdf
#
# 4. تشغيل الخادم للوصول من الجوال (نفس الشبكة):
#    python run.py --local --host 0.0.0.0
#    # ثم افتح من الجوال: http://192.168.1.X:7860
#
# 5. إعداد المزامنة مع Syncthing:
#    sudo pacman -S syncthing
#    systemctl --user enable --now syncthing
#    # افتح http://127.0.0.1:8384 وأضف مجلد المشروع
#
# 6. تفعيل CUDA (إذا كان لديك NVIDIA GPU):
#    pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
#
print("Local mode guide - see cell source for commands")